# colab_01 — Li 2025: download + per-study QC

Vertical-slice smoke test on the smallest of the three substrate studies: **Li 2025**, GEO GSE237718, temporal cortex, 56 donors, 343k nuclei, purpose-built E2/E3-3/E4 carrier matching. Tests the end-to-end pipeline (download → load → secondary QC → save → audit) on a study that fits cheaply, before scaling to SEA-AD (1.2M) and Brase 2023 (294k).

**Order of operations:**

1. Setup — mount Drive, clone/pull repo, install integration env (section 1)
2. Env capture for this notebook (section 2)
3. Download Li 2025 from GEO into runtime-local cache (section 3)
4. Load into AnnData; print shape + obs/var sanity (section 4)
5. Secondary QC with locked defaults — mito %, filter cells/genes, Scrublet per sample, mito sensitivity sweep (section 5)
6. Audit A partial — niche-critical gene presence in `var_names` (section 6)
7. Save processed AnnData to Drive; cleanup raw if Audit F pressure (section 7)
8. Handoff to `colab_02` (next study) (section 8)

Audit C (Brase DIAN filter) does **not** apply to Li 2025 — only Brase 2023 mixes DIAN + Knight ADRC cohorts.

## 1 — Setup

Same pattern as `colab_00`: Drive mount, runtime-local repo clone, `requirements_integration.txt` install.

### 1a — Mount Drive + clone/pull repo + install env

Asserts Python 3.12 before installing — scvi-tools 1.4.x requires it. Adds repo to `sys.path` so `from src.* import ...` resolves.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info[:2]}"

!pip install -q -r {REPO_PATH}/requirements_integration.txt

## 2 — Environment capture

Per-notebook env snapshot. Same shape as `colab_00`'s section 3 but condensed into one cell since the pattern is now established.

### 2a — pip freeze + env JSON

Writes `colab_01_<date>_pip_freeze.txt` and `colab_01_<date>_env.json` under `outputs/software_versions/`. Re-captures because Colab runtime resets can produce different resolved versions session to session.

In [ ]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_01"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

env_snapshot = {
    "notebook_id":      NOTEBOOK_ID,
    "date":             TODAY,
    "python_version":   sys.version,
    "platform":         platform.platform(),
    "os_release":       platform.release(),
    "gpu":              _run(["nvidia-smi", "-L"]),
    "nvidia_driver":    _run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]),
    "git_commit":       _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
}
try:
    import torch
    env_snapshot["torch_version"]      = torch.__version__
    env_snapshot["torch_cuda_version"] = torch.version.cuda
except ImportError:
    env_snapshot["torch_version"] = None

ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))

## 3 — Download Li 2025

GEO accession **GSE237718**. The series' supplementary directory follows the standard GEO path `https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237nnn/GSE237718/suppl/`. Files there could be (depending on what authors deposited): a single aggregated `.h5ad` / `.rds`, per-sample 10x outputs (`matrix.mtx.gz` + `barcodes.tsv.gz` + `features.tsv.gz`), or `.h5` HDF5 dumps. The notebook fetches everything in `suppl/`, then section 4 branches on what landed.

Raw download lives under runtime-local `/content/li2025_raw/` (not Drive) — it's deleted at end of section 7 per Audit F's keep=False ruling.

### 3a — Fetch GEO supplementary directory

Recursive `wget` of the GEO suppl directory. Time depends on size of deposited matrices; expect several minutes.

In [ ]:
GEO_ACCESSION = "GSE237718"
GEO_SUPPL_URL = f"https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237nnn/{GEO_ACCESSION}/suppl/"
RAW_DIR = "/content/li2025_raw"
os.makedirs(RAW_DIR, exist_ok=True)

!wget -q -r -np -nH --cut-dirs=5 --reject "index.html*" -P {RAW_DIR} {GEO_SUPPL_URL}
!ls -lh {RAW_DIR}

## 4 — Load Li 2025 into AnnData

Branches on file format. Three common GEO deposit shapes are handled:

- **Aggregated h5ad / h5** — single file, load directly with `sc.read_h5ad` / `sc.read_hdf`.
- **Per-sample 10x triples** — directory layout, `sc.read_10x_mtx` per sample, then `anndata.concat`.
- **Single `matrix.mtx.gz` + barcodes + features** — `sc.read_10x_mtx` directly.

If the deposit doesn't match any of these, the load cell will raise and you should inspect the file listing manually.

### 4a — Detect deposit format and load

Sets `adata` (the raw AnnData). Logs which format branch was taken. Does NOT yet attach donor-level APOE metadata — that's 4b.

In [ ]:
import glob
import anndata as ad
import scanpy as sc

h5ad_files = sorted(glob.glob(f"{RAW_DIR}/**/*.h5ad", recursive=True))
h5_files   = sorted(glob.glob(f"{RAW_DIR}/**/*.h5",   recursive=True))
mtx_files  = sorted(glob.glob(f"{RAW_DIR}/**/matrix.mtx.gz", recursive=True))

print(f"h5ad: {len(h5ad_files)}  h5: {len(h5_files)}  mtx: {len(mtx_files)}")

if h5ad_files:
    adata = sc.read_h5ad(h5ad_files[0])
    LOAD_BRANCH = f"h5ad ({h5ad_files[0]})"
elif h5_files:
    adata = sc.read_10x_h5(h5_files[0])
    LOAD_BRANCH = f"10x h5 ({h5_files[0]})"
elif len(mtx_files) == 1:
    adata = sc.read_10x_mtx(os.path.dirname(mtx_files[0]))
    LOAD_BRANCH = f"single 10x mtx ({os.path.dirname(mtx_files[0])})"
elif len(mtx_files) > 1:
    parts = []
    for m in mtx_files:
        sample_dir = os.path.dirname(m)
        a = sc.read_10x_mtx(sample_dir)
        a.obs["sample_id"] = os.path.basename(sample_dir)
        parts.append(a)
    adata = ad.concat(parts, axis=0, join="outer", merge="unique")
    LOAD_BRANCH = f"per-sample 10x mtx, {len(parts)} samples concatenated"
else:
    raise RuntimeError(f"Unrecognized GEO deposit shape in {RAW_DIR}; inspect manually.")

adata.var_names_make_unique()
print(f"Load branch: {LOAD_BRANCH}")
print(adata)

### 4b — Attach donor-level metadata

Donor-level APOE genotype is the design's secondary stratification axis. If GEO ships a sample/donor manifest (often `*_metadata.csv.gz` or similar), merge it onto `adata.obs`. If not, leave a TODO — the paper's Table S1 may have donor APOE assignments that need to be downloaded separately.

Result of this cell: `adata.obs` should carry at minimum a `donor_id` column and ideally an `apoe_genotype` column.

In [ ]:
import pandas as pd

# TODO — confirm the metadata filename pattern after section 3a lists the deposit.
metadata_candidates = (
    sorted(glob.glob(f"{RAW_DIR}/**/*metadata*.csv*", recursive=True))
    + sorted(glob.glob(f"{RAW_DIR}/**/*metadata*.tsv*", recursive=True))
    + sorted(glob.glob(f"{RAW_DIR}/**/*sample*.csv*",   recursive=True))
)
print("metadata candidates:", metadata_candidates)

if metadata_candidates:
    meta_df = pd.read_csv(metadata_candidates[0], sep=None, engine="python")
    print(meta_df.head())
    print("columns:", list(meta_df.columns))
    # TODO — set merge key after inspecting columns (likely 'sample_id' / 'cell_barcode' / 'donor_id').
    MERGE_LEFT  = None  # e.g. "sample_id"
    MERGE_RIGHT = None  # e.g. "sample_id"
    if MERGE_LEFT and MERGE_RIGHT:
        adata.obs = adata.obs.merge(meta_df, left_on=MERGE_LEFT, right_on=MERGE_RIGHT, how="left")
else:
    print("No sample/donor metadata in GEO deposit. Look up Li 2025 Table S1 separately.")

print("obs columns:", list(adata.obs.columns))
print("obs head:")
print(adata.obs.head())

## 5 — Secondary QC (locked defaults)

Per the locked spec (2026-05-21) and `src/qc.py`:

- min 500 counts / nucleus
- min 250 genes / nucleus
- max 5% mito (tight because **nuclei** — high mito = cytoplasmic contamination)
- min 10 cells / gene
- Scrublet per sample
- Mito threshold sensitivity check at 3% / 5% / 10%

Two-layer principle: Li 2025's published per-study QC + cell-type labels are inherited (if shipped on GEO); this is the **secondary** uniform filter on top.

Implemented inline in scanpy; will be lifted into `src/qc.py:secondary_qc` after the pattern stabilizes across all three studies.

### 5a — Annotate mito % and QC metrics

Detects var_name format (HGNC `MT-*` vs Ensembl `ENSG...`) and computes `pct_counts_mt` accordingly. Logs n_cells, n_genes, mito gene count for sanity.

In [ ]:
looks_ensembl = adata.var_names.str.startswith("ENSG").mean() > 0.5
if looks_ensembl:
    # TODO — Li 2025 var_names appear to be Ensembl IDs. Need a gene_symbols column
    # on adata.var (often shipped by GEO as 'gene_symbols' or 'feature_name') to
    # detect mito genes via 'MT-' prefix. Set the column name here:
    SYMBOL_COL = None  # e.g. "gene_symbols"
    if SYMBOL_COL and SYMBOL_COL in adata.var.columns:
        adata.var["mt"] = adata.var[SYMBOL_COL].astype(str).str.startswith("MT-")
    else:
        raise RuntimeError(
            "var_names are Ensembl but no gene_symbols column identified. "
            "Inspect adata.var.columns and set SYMBOL_COL."
        )
else:
    adata.var["mt"] = adata.var_names.str.startswith("MT-")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None, log1p=False)

print(f"shape:        {adata.shape}")
print(f"var format:   {'Ensembl' if looks_ensembl else 'HGNC symbols'}")
print(f"mito genes:   {int(adata.var['mt'].sum())}")
print(adata.obs[["total_counts", "n_genes_by_counts", "pct_counts_mt"]].describe())

### 5b — Apply locked thresholds

Filters in order: cells by counts, cells by gene count, cells by mito %, then genes by min cell count. Records before/after counts for the audit report.

In [ ]:
QC_PARAMS = {
    "min_counts":         500,
    "min_genes":          250,
    "max_mito_pct":       5.0,
    "min_cells_per_gene": 10,
}

n_cells_0, n_genes_0 = adata.shape

sc.pp.filter_cells(adata, min_counts=QC_PARAMS["min_counts"])
n_after_counts = adata.n_obs

sc.pp.filter_cells(adata, min_genes=QC_PARAMS["min_genes"])
n_after_genes = adata.n_obs

adata = adata[adata.obs["pct_counts_mt"] < QC_PARAMS["max_mito_pct"]].copy()
n_after_mito = adata.n_obs

sc.pp.filter_genes(adata, min_cells=QC_PARAMS["min_cells_per_gene"])

qc_trace = {
    "params": QC_PARAMS,
    "n_cells_initial":          n_cells_0,
    "n_cells_after_min_counts": n_after_counts,
    "n_cells_after_min_genes":  n_after_genes,
    "n_cells_after_max_mito":   n_after_mito,
    "n_cells_final":            adata.n_obs,
    "n_genes_initial":          n_genes_0,
    "n_genes_final":            adata.n_vars,
}
print(json.dumps(qc_trace, indent=2))

### 5c — Scrublet per sample

Doublet detection is run **per sample**, not on the concatenated object, because doublet rate is a per-library property. Requires a sample-level column on `obs` — defaults to `sample_id` if available, otherwise prompts for an alternative.

In [ ]:
import numpy as np
import scrublet as scr

candidate_sample_cols = ["sample_id", "library_id", "batch", "sample", "orig.ident"]
SAMPLE_COL = next((c for c in candidate_sample_cols if c in adata.obs.columns), None)
if SAMPLE_COL is None:
    raise RuntimeError(
        f"No sample-level column on obs. Candidates checked: {candidate_sample_cols}. "
        "Set SAMPLE_COL manually after inspecting adata.obs.columns."
    )
print(f"Scrublet per sample using column: {SAMPLE_COL}")

adata.obs["doublet_score"]    = np.nan
adata.obs["predicted_doublet"] = False

for sample in adata.obs[SAMPLE_COL].unique():
    mask = adata.obs[SAMPLE_COL] == sample
    sub = adata[mask]
    if sub.n_obs < 30:
        print(f"  {sample}: n={sub.n_obs} — too small, skipping")
        continue
    sclt = scr.Scrublet(sub.X)
    scores, calls = sclt.scrub_doublets(verbose=False)
    adata.obs.loc[mask, "doublet_score"]    = scores
    adata.obs.loc[mask, "predicted_doublet"] = calls
    print(f"  {sample}: n={sub.n_obs}  doublets={int(np.sum(calls))} ({100*np.mean(calls):.1f}%)")

n_before_doublet = adata.n_obs
adata = adata[~adata.obs["predicted_doublet"].astype(bool)].copy()
qc_trace["n_cells_after_scrublet"] = adata.n_obs
qc_trace["scrublet_removed"]       = int(n_before_doublet - adata.n_obs)
qc_trace["sample_col"]              = SAMPLE_COL
print(f"After Scrublet: {adata.n_obs} cells ({qc_trace['scrublet_removed']} removed)")

### 5d — Mito threshold sensitivity (3% / 5% / 10%)

Sanity check for the locked 5% default. If astro + microglia counts shift dramatically across 3% / 5% / 10%, the 5% default is load-bearing and needs justification. Recomputes from the **pre-QC** cell pool (need to reload or re-derive — for this notebook we recompute against the pct_counts_mt that was annotated on the original object before filtering).

Caveat: this cell needs the pre-filter `pct_counts_mt` distribution. Easiest is to re-derive from `adata` (post-filter) which can only tell us cells remaining at the chosen threshold. A proper sensitivity sweep requires the pre-filter distribution — drafted as a TODO for now and run properly in `colab_03_integration` where the concatenated pre-QC object is constructed.

In [ ]:
# TODO — proper sensitivity sweep needs pre-filter pct_counts_mt distribution.
# Run as a quick sanity check by reloading raw + recomputing for 3 thresholds.
# Deferred: full implementation lives in colab_03_integration on the concatenated object.
qc_trace["mito_sensitivity"] = {"status": "deferred-to-colab_03"}
print(qc_trace["mito_sensitivity"])

## 6 — Audit A (partial): niche-critical gene presence

Full Audit A is FM-vocab intersection (Geneformer Ensembl + scGPT HGNC) — that needs the FM env (Python 3.10, see `requirements_fm.txt`) and lives in a later notebook.

This partial Audit A asks one question: are the niche-critical genes present in Li 2025's `var_names` (or `gene_symbols`) at all? If any are missing here, they will definitely be missing downstream — fail early.

Niche-critical (locked 2026-05-22): `APOE`, `TREM2`, `MS4A6A`, `CLU`, `GFAP`, `AQP4`, `AIF1` (IBA1), `CSF1R`.

### 6a — Check presence and write Audit A (Li 2025 row) to audit_report.json

Searches `var_names` and the `gene_symbols` column (if Ensembl-named). Reports per-gene status. Final Audit A pass-criterion (zero dropped, across all studies × both FMs) is only computable after all three studies + FM vocab intersection — this writes only the Li 2025 row.

In [ ]:
from src.data_loaders import NICHE_CRITICAL_GENES

if looks_ensembl and SYMBOL_COL and SYMBOL_COL in adata.var.columns:
    gene_pool = set(adata.var[SYMBOL_COL].astype(str))
else:
    gene_pool = set(adata.var_names.astype(str))

per_gene = {g: (g in gene_pool) for g in NICHE_CRITICAL_GENES}
missing  = [g for g, present in per_gene.items() if not present]

audit_a_li2025 = {
    "study":           "Li 2025",
    "vocab_format":    "Ensembl" if looks_ensembl else "HGNC",
    "symbol_col_used": SYMBOL_COL if looks_ensembl else None,
    "n_genes_post_qc": adata.n_vars,
    "per_gene":        per_gene,
    "missing":         missing,
    "status":          "pass" if not missing else "fail",
    "note":            "partial — FM-vocab intersection deferred to FM env notebook",
}

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
if os.path.exists(AUDIT_REPORT_PATH):
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
else:
    report = {}
audit_a = report.get("audit_a_vocab_niche_critical", {"per_study": {}})
audit_a["per_study"]["Li 2025"] = audit_a_li2025
audit_a["status"] = "partial"
report["audit_a_vocab_niche_critical"] = audit_a
report["li2025_qc_trace"]              = qc_trace
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(audit_a_li2025, indent=2))

## 7 — Save processed AnnData and clean up raw

Processed h5ad goes to Drive (`*.h5ad` is gitignored — it lives outside the repo). Raw GEO download is deleted per Audit F (`keep=False`) — saves several GB.

### 7a — Save Li 2025 processed AnnData

Sparse CSR enforced before write — memory discipline from the integration-env compute constraints (Colab high-RAM ca. 50 GB, concatenated multi-study tight).

In [ ]:
from scipy.sparse import csr_matrix, issparse

if not issparse(adata.X) or adata.X.getformat() != "csr":
    adata.X = csr_matrix(adata.X)

PROCESSED_DIR  = os.path.join(DRIVE_ROOT, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
PROCESSED_PATH = os.path.join(PROCESSED_DIR, "li2025_processed.h5ad")
adata.write_h5ad(PROCESSED_PATH, compression="gzip")

size_mb = os.path.getsize(PROCESSED_PATH) / 1024**2
print(f"Wrote {PROCESSED_PATH}  ({size_mb:.1f} MB)")

### 7b — Delete raw download

Per Audit F: raw downloads are `keep=False`, deleted after per-study processed AnnData saved. Saves the GEO supplementary footprint (several GB) back.

In [ ]:
import shutil

raw_size_gb = sum(os.path.getsize(os.path.join(dp, f))
                  for dp, _, fs in os.walk(RAW_DIR) for f in fs) / 1024**3
print(f"Raw dir size before delete: {raw_size_gb:.2f} GB")

shutil.rmtree(RAW_DIR)
print(f"Deleted {RAW_DIR}")

## 8 — Handoff to colab_02

Prints audit summary + git commit instructions. If Li 2025 is clean here, the same pattern scales to SEA-AD (colab_02) and Brase 2023 (colab_03) — Brase additionally runs Audit C (DIAN filter).

### 8a — Summary + commit instructions

Lists artifacts written and the explicit `git add / commit / push` commands. Processed h5ad is gitignored (lives only on Drive); freeze, env JSON, and audit_report are committed.

In [ ]:
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

print("=== Audit summary ===")
for k, v in report.items():
    print(f"  {k}: {v.get('status') if isinstance(v, dict) else v}")

print("\n=== Artifacts ===")
print(f"  committable:  {FREEZE_PATH}")
print(f"  committable:  {ENV_JSON_PATH}")
print(f"  committable:  {AUDIT_REPORT_PATH}")
print(f"  drive-only:   {PROCESSED_PATH}")

rel_paths = [os.path.relpath(p, REPO_PATH) for p in [FREEZE_PATH, ENV_JSON_PATH, AUDIT_REPORT_PATH]]
msg = f"colab_01: Li 2025 QC + Audit A (partial) ({TODAY})"
print("\n=== Commit + push ===")
for c in [
    f"cd {REPO_PATH} && git add {' '.join(rel_paths)}",
    f"cd {REPO_PATH} && git commit -m {msg!r}",
    f"cd {REPO_PATH} && git push",
]:
    print(f"  {c}")

print("\nNext: colab_02_seaad_qc.ipynb (SEA-AD download + secondary_qc + Audit A row).")